# Score-based search (XGES / hill climbing / Tabu search)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/score_search.ipynb)

XGES, hill climbing, and Tabu search look for the graph with the best BIC score. XGES searches
over equivalence classes, as GES does, with deletions first. Hill climbing adds, removes, or
reverses one edge of a DAG at a time and stops when no move improves the score; Tabu search keeps
going past that point and returns the best DAG it visits. All three return a CPDAG and suit
linear-Gaussian data.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.6.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.6.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 2,000 samples.

In [3]:
seed = 7
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

2000 observations, 10 variables, seed 7


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 332" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="332" fill="#fafbfc"/><line x1="312.82" y1="44.82" x2="242.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,39.30 310.06,40.65 315.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="144.93" y1="163.00" x2="409.13" y2="134.59" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.07,133.52 408.60,129.62 409.67,139.56" fill="#3a4658" stroke-linejoin="round"/><line x1="113.24" y1="177.70" x2="42.60" y2="227.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,171.91 110.35,173.62 116.14,181.78" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="151.70" x2="42.84" y2="105.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,157.22 115.57,147.53 110.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="409.32" y1="128.29" x2="244.82" y2="100.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.18,129.96 410.16,123.36 408.49,133.22" fill="#3a4658" stroke-linejoin="round"/><line x1="410.18" y1="124.86" x2="144.33" y2="36.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.67,128.02 411.76,120.11 408.60,129.60" fill="#3a4658" stroke-linejoin="round"/><line x1="410.11" y1="139.19" x2="344.37" y2="160.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.63,136.12 408.58,134.43 411.65,143.95" fill="#3a4658" stroke-linejoin="round"/><line x1="139.66" y1="224.85" x2="218.45" y2="116.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="224.34,108.76 214.41,113.89 222.49,119.79" fill="#3a4658" stroke-linejoin="round"/><line x1="142.92" y1="242.40" x2="212.67" y2="287.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.08,292.82 215.38,283.20 209.96,291.60" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="235.35" x2="45.00" y2="235.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,235.35 109.00,230.35 109.00,240.35" fill="#3a4658" stroke-linejoin="round"/><line x1="212.82" y1="85.57" x2="142.84" y2="39.30" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.16,91.09 215.57,81.40 210.06,89.74" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="105.43" x2="312.82" y2="151.70" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,157.22 315.57,147.53 310.06,155.87" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="98.26" x2="45.00" y2="98.26" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,98.26 209.00,93.26 209.00,103.26" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="44.82" x2="42.84" y2="91.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,39.30 110.06,40.65 115.57,48.99" fill="#3a4658" stroke-linejoin="round"/><line x1="210.11" y1="292.81" x2="44.37" y2="239.35" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.63,295.88 211.65,288.05 208.58,297.57" fill="#3a4658" stroke-linejoin="round"/><circle cx="332.00" cy="32.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="32.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="132.00" cy="164.39" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="164.39" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="432.00" cy="132.13" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="132.13" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominan

## Fit XGES, hill climbing, and Tabu search

`metadata["score"]` is the BIC score of the returned graph, which each search minimizes. A larger
`lambda_value` weighs the BIC penalty more and gives a sparser graph. With
`extended_search=False`, XGES skips its extended search for a faster fit, and `tabu_length` sets
how many recently visited DAGs Tabu search may not return to.

In [4]:
results = {
    "XGES": andrey.xges(X),
    "Hill climbing": andrey.hc(X),
    "Tabu search": andrey.tabu(X),
}
for name, result in results.items():
    print(f"{name}: {result.structure.kind}, score {result.metadata['score']:.1f}")
    display(andrey.viz.draw(result.structure))

XGES: cpdag, score -22406.2


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 279" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="279" fill="#fafbfc"/><line x1="209.35" y1="242.74" x2="144.80" y2="231.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.20,244.49 210.23,237.82 208.48,247.67" fill="#3a4658" stroke-linejoin="round"/><line x1="240.90" y1="43.33" x2="316.25" y2="123.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.10,130.83 319.90,120.12 312.61,126.97" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.85" x2="145.00" y2="33.85" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.85 209.00,28.85 209.00,38.85" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.55" x2="42.00" y2="191.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.16 211.11,44.70 217.50,52.39" fill="#3a4658" stroke-linejoin="round"/><path d="M 313.70 154.24 Q 242.54 208.43 144.78 226.64" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.66,148.18 310.67,150.27 316.73,158.22" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="132.62" x2="244.25" y2="109.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,135.96 312.00,127.90 308.65,137.33" fill="#3a4658" stroke-linejoin="round"/><line x1="310.32" y1="148.00" x2="244.25" y2="171.45" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.75,144.66 308.65,143.29 312.00,152.71" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.38" x2="117.69" y2="211.01" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.84 121.60,207.90 113.77,214.12" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="103.31" x2="209.00" y2="104.63" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.72 209.04,99.63 208.96,109.63" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.80" x2="113.10" y2="46.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.26 110.25,42.85 115.95,51.07" fill="#3a4658" stroke-linejoin="round"/><line x1="140.15" y1="218.89" x2="217.58" y2="122.73" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.95 213.68,119.60 221.47,125.87" fill="#3a4658" stroke-linejoin="round"/><line x1="143.48" y1="222.91" x2="211.70" y2="186.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.52,181.90 209.35,182.19 214.05,191.01" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.61" x2="44.48" y2="203.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.39 111.31,217.80 108.52,227.41" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.45" x2="109.00" y2="33.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.64 109.08,28.48 108.92,38.48" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="246.76" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="246.76" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="33.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="332.00" cy="140.31" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="140.31" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.21" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.21" text-anchor="middle" font-size="11" font-weight="600

Hill climbing: cpdag, score -22342.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 764 398" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="764" height="398" fill="#fafbfc"/><line x1="45.00" y1="65.94" x2="119.00" y2="65.94" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="236.99" y1="260.38" x2="323.16" y2="53.30" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="327.01,44.07 318.55,51.38 327.78,55.22" fill="#3a4658" stroke-linejoin="round"/><line x1="244.87" y1="274.25" x2="309.24" y2="283.60" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.13,285.04 309.96,278.65 308.52,288.55" fill="#3a4658" stroke-linejoin="round"/><line x1="221.97" y1="251.69" x2="137.67" y2="77.64" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="226.33,260.69 226.47,249.51 217.47,253.87" fill="#3a4658" stroke-linejoin="round"/><line x1="241.84" y1="263.89" x2="414.59" y2="114.83" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="422.16,108.30 411.32,111.05 417.85,118.62" fill="#3a4658" stroke-linejoin="round"/><line x1="215.20" y1="288.10" x2="141.49" y2="357.05" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.51,281.27 211.79,284.45 218.62,291.75" fill="#3a4658" stroke-linejoin="round"/><path d="M 244.97 271.50 Q 433.57 258.53 609.41 292.25" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.23,294.13 610.35,287.34 608.47,297.16" fill="#3a4658" stroke-linejoin="round"/><line x1="243.27" y1="265.90" x2="512.06" y2="111.28" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="520.73,106.29 509.57,106.94 514.56,115.61" fill="#3a4658" stroke-linejoin="round"/><path d="M 244.97 271.50 Q 481.79 255.36 709.01 263.50" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="719.01,263.86 709.19,258.50 708.84,268.50" fill="#3a4658" stroke-linejoin="round"/><line x1="309.32" y1="35.90" x2="144.82" y2="63.76" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.18,34.24 308.49,30.98 310.16,40.83" fill="#3a4658" stroke-linejoin="round"/><line x1="342.76" y1="39.36" x2="421.24" y2="92.52" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="344.31" y1="36.24" x2="519.69" y2="95.64" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="316.57" y1="269.85" x2="140.72" y2="75.57" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.28,277.26 320.27,266.50 312.86,273.21" fill="#3a4658" stroke-linejoin="round"/><line x1="338.13" y1="275.44" x2="425.87" y2="111.27" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="344.99" y1="287.32" x2="609.01" y2="295.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.01,296.16 609.17,290.84 608.85,300.84" fill="#3a4658" stroke-linejoin="round"/><line x1="344.98" y1="286.17" x2="709.04" y2="265.62" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="719.02,265.06 708.75,260.63 709.32,270.61" fill="#3a4658" stroke-linejoin="round"/><line x1="144.92" y1="67.39" x2="409.15" y2="97.23" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.08,98.35 409.71,92.26 408.58,102.19" fill="#3a4658" stroke-linejoin="round"/><path d="M 144.99 65.36 Q 334.19 56.96 509.52 94.94" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.29,97.05 510.58,90.05 508.46,99.82" fill="#3a4658" stroke-linejoin="round"/><line x1="144.34" y1="70.02" x2="710.16" y2="257.10" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="719.66,260.24 711.73,252.35 708.59,261.85" fill="#3a4658" stroke-linejoin="round"/><line x1="445.00" y1="99.81" x2="519.00" y2="99.81" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="144.88" y1="364.15" x2="609.22" y

Tabu search: cpdag, score -22398.8


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 333" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="333" fill="#fafbfc"/><line x1="45.00" y1="225.37" x2="119.00" y2="225.37" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="340.97" y1="107.36" x2="416.13" y2="186.15" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="423.03,193.38 419.74,182.70 412.51,189.60" fill="#3a4658" stroke-linejoin="round"/><line x1="310.16" y1="90.73" x2="144.34" y2="35.90" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.66,93.87 311.73,85.98 308.59,95.48" fill="#3a4658" stroke-linejoin="round"/><line x1="309.00" y1="97.95" x2="245.00" y2="97.95" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,97.95 309.00,92.95 309.00,102.95" fill="#3a4658" stroke-linejoin="round"/><line x1="409.06" y1="204.52" x2="144.96" y2="224.40" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.04,203.77 408.69,199.53 409.44,209.50" fill="#3a4658" stroke-linejoin="round"/><line x1="409.32" y1="198.95" x2="244.82" y2="171.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.18,200.62 410.16,194.02 408.49,203.88" fill="#3a4658" stroke-linejoin="round"/><line x1="415.60" y1="218.92" x2="341.27" y2="292.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="422.73,211.91 412.10,215.36 419.11,222.49" fill="#3a4658" stroke-linejoin="round"/><line x1="41.34" y1="137.64" x2="122.66" y2="216.33" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="44.74" y1="131.17" x2="219.26" y2="166.35" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="41.34" y1="119.56" x2="115.47" y2="47.82" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="122.66,40.86 112.00,44.22 118.95,51.41" fill="#3a4658" stroke-linejoin="round"/><line x1="44.85" y1="126.63" x2="219.15" y2="99.92" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="143.32" y1="218.98" x2="220.68" y2="175.31" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="144.16" y1="229.98" x2="319.84" y2="296.57" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="140.03" y1="215.14" x2="223.97" y2="108.18" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="45.00" y1="31.82" x2="109.00" y2="31.82" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,31.82 109.00,26.82 109.00,36.82" fill="#3a4658" stroke-linejoin="round"/><circle cx="32.00" cy="225.37" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="225.37" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="332.00" cy="97.95" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="332.00" y="97.95" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="432.00" cy="202.79" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="202.79" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="128.60" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="128.60" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="132.00" cy="225.37" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="225.37" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="232.00" cy="168.92" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="168.92" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">5

## Compare with GES over ten datasets

Structural Hamming distance (SHD) counts errors against the truth's CPDAG; lower is better. The
empty graph's SHD is the number of true edges, so an estimate at or above it has not improved on
returning no edges. One dataset can favor any method, so the table covers ten seeds and their
median.

In [5]:
methods = {
    "GES": lambda X: andrey.ges(X),
    "XGES": lambda X: andrey.xges(X),
    "Hill climbing": lambda X: andrey.hc(X),
    "Tabu search": lambda X: andrey.tabu(X),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=s, scale="standardize")
    d = draw.observed_graph.n_nodes
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="cpdag")
    rows[s] = {"Empty graph": metrics.score(empty, draw.observed_graph)["shd"]}
    for name, fit in methods.items():
        rows[s][name] = metrics.score(fit(draw.data), draw.observed_graph)["shd"]
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

seed,0,1,2,3,4,5,6,7,8,9,median
Empty graph,15,15,15,15,15,15,15,15,15,15,15.0
GES,21,10,0,10,17,7,11,17,0,0,10.0
XGES,0,8,0,9,1,0,1,6,0,0,0.5
Hill climbing,22,22,0,19,1,13,22,22,19,4,19.0
Tabu search,21,0,0,9,1,0,16,11,22,4,6.5


Tabu search starts with hill climbing's moves, so its score is never worse than hill
climbing's on the same data. The [0.3.0 release card](https://andrey.abel.ai/docs/benchmarks/0.3.0.html)
compares XGES with the xges package and the other two with bnlearn, on larger graphs with ten rows
per variable.

## Use another dataset

Change `seed`, or the `d` and `n` passed to `benchmarks.scm` and `sample`, and rerun.